# Interactive Entry Labeling

Label desirable bullish entry candles inside deterministic rolling windows. The planned window controls which candles are written and where **Next** navigates; Plotly zooming and panning never change labeling coverage or progress.

The notebook intentionally excludes the locked test period. Set `LABELING_END_DATE` to the inclusive end of validation before creating a session. Forward outcomes are only calculated when the complete configured horizon also remains before that boundary.

In [ ]:
from __future__ import annotations

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from swingtrader.core.paths import find_repo_root
from swingtrader.data.bronze.queries import load_available_tickers
from swingtrader.data.db import resolve_database_engine
from swingtrader.modeling.labeling import (
    CHART_TIMEFRAME_SESSIONS,
    DEFAULT_CHART_TIMEFRAME,
    LabelingConfig,
    LabelingSession,
    PreparedLabelingUniverse,
    build_labeling_figure,
    create_labeling_session,
    initialize_labeling_tables,
    load_labeling_session,
    load_labels,
    load_latest_labeling_session,
    prepare_chart_view,
    prepare_labeling_universe,
    risk_guide_for_date,
    save_labeling_window,
    update_price_hud,
    update_risk_guide_traces,
    update_selected_trace,
)

# Resolve an absolute database URL so the notebook always targets the repository
# database, regardless of the kernel's working directory. Without this, the
# default relative "data/swingtrader.sqlite" path resolves next to the notebook
# and silently creates an empty database.
repo_root = find_repo_root()
DATABASE_URL = f"sqlite+pysqlite:///{(repo_root / 'data' / 'swingtrader.sqlite').as_posix()}"


## Session configuration

`WINDOW_SIZE` and `STEP_SIZE` are fixed when a session is created. A typical `80/60` configuration makes three quarters of each next window new while retaining 20 sessions of overlap. The ATR stop multiple and reward/risk ratio remain adjustable in the UI because they are visual calibration aids.

In [ ]:
PROVIDER = "yfinance"
TICKERS = load_available_tickers(
    engine=resolve_database_engine(database_url=DATABASE_URL),
    provider=PROVIDER,
)
LABEL_FAMILY = "trend_continuation"
LABELING_SESSION_ID: str | None = None
RESUME_LATEST = True
LABELING_START_DATE: str | None = None
LABELING_END_DATE: str | None = "2022-01-01"  # Required for a new session: validation end, never test end.

NEW_SESSION_CONFIG = LabelingConfig(
    window_size=80,
    step_size=60,
    forward_horizon=10,
    atr_length=14,
    atr_stop_multiple=1.5,
    reward_risk_ratio=3.0,
    commission_rate=0.0025,
    pivot_high_left=15,
    pivot_high_right=15,
    pivot_low_left=15,
    pivot_low_right=15,
    default_heatmap_mode="net_return",
)


In [ ]:
# import swingtrader.modeling.labeling as labeling

# engine = resolve_database_engine(database_url=DATABASE_URL)

# # Drop only the two labeling tables.
# labeling.metadata.drop_all(engine, tables=[labeling.candle_labels, labeling.labeling_sessions])

In [ ]:
engine = resolve_database_engine(database_url=DATABASE_URL)
initialize_labeling_tables(engine)

if LABELING_SESSION_ID is not None:
    session = load_labeling_session(
        engine=engine, labeling_session_id=LABELING_SESSION_ID
    )
elif RESUME_LATEST:
    session = load_latest_labeling_session(
        engine=engine, provider=PROVIDER, label_family=LABEL_FAMILY
    )
else:
    session = None

if session is None:
    if LABELING_END_DATE is None:
        raise ValueError(
            "Set LABELING_END_DATE to the inclusive validation end before creating a session."
        )
    universe = prepare_labeling_universe(
        engine=engine,
        provider=PROVIDER,
        tickers=TICKERS,
        labeling_start_date=LABELING_START_DATE,
        labeling_end_date=LABELING_END_DATE,
        config=NEW_SESSION_CONFIG,
    )
    session = create_labeling_session(
        engine=engine,
        provider=PROVIDER,
        tickers=universe.tickers,
        label_family=LABEL_FAMILY,
        labeling_start_date=LABELING_START_DATE,
        labeling_end_date=LABELING_END_DATE,
        config=NEW_SESSION_CONFIG,
    )
    print(
        f"Created {session.labeling_session_id} with "
        f"{len(universe.tickers)} eligible tickers."
    )
    if universe.skipped_tickers:
        print(f"Skipped {len(universe.skipped_tickers)} ineligible tickers:")
        for ticker, reason in universe.skipped_tickers.items():
            print(f"- {ticker}: {reason}")
else:
    requested_start = (
        None
        if LABELING_START_DATE is None
        else pd.Timestamp(LABELING_START_DATE).date()
    )
    requested_end = (
        None
        if LABELING_END_DATE is None
        else pd.Timestamp(LABELING_END_DATE).date()
    )
    mismatches = []
    if not set(session.tickers).issubset(TICKERS):
        mismatches.append("tickers")
    if session.labeling_start_date != requested_start:
        mismatches.append("labeling_start_date")
    if requested_end is not None and session.labeling_end_date != requested_end:
        mismatches.append("labeling_end_date")
    if session.config != NEW_SESSION_CONFIG:
        mismatches.append("configuration")
    if mismatches:
        raise ValueError(
            "The resumed session does not match the requested " + ", ".join(mismatches)
        )

    universe = prepare_labeling_universe(
        engine=engine,
        provider=session.provider,
        tickers=session.tickers,
        labeling_start_date=session.labeling_start_date,
        labeling_end_date=session.labeling_end_date,
        config=session.config,
    )
    if universe.skipped_tickers:
        skipped = ", ".join(universe.skipped_tickers)
        raise ValueError(
            "The persisted session contains tickers that are no longer eligible: "
            f"{skipped}. Create a new session with RESUME_LATEST = False."
        )
    print(
        "Resuming",
        session.labeling_session_id,
        session.current_ticker,
        f"window {session.current_window_position + 1}",
    )

session


## Interactive workflow

- Click a candle to toggle its positive label.
- Unselected candles in the planned window are saved as negatives.
- Candles reached only by panning outside the planned window remain unlabeled.
- **Timeframe** (`4M`, `1Y`, `3Y`) sets how many trading sessions are visible. The active window's centre always stays at the centre of the figure with balanced context on each side (for example `3Y` shows ~1.5 years to the left and ~1.5 years to the right). The shaded regions mark everything outside the active labeling window. `4M` (~84 sessions) is the default.
- Hover a candle to display close-based ATR stop and take-profit guides.
- **Reset labels** restores the state loaded for the current window without writing.
- **Save** commits the current window without moving.
- **Next** saves atomically and advances according to the planned window sequence, regardless of the current Plotly viewport.


In [ ]:
TIMEFRAME_BUTTON_STYLE = widgets.HTML(
    "<style>"
    ".labeling-timeframe .widget-toggle-button {"
    "background-color: #ffffff; color: #000000; border: 1px solid #000000;"
    "border-radius: 8px; margin: 0 3px; padding: 0 10px; box-shadow: none;"
    "width: auto;}"
    ".labeling-timeframe .widget-toggle-button:hover {background-color: #f0f0f0;}"
    ".labeling-timeframe .widget-toggle-button.mod-active {"
    "background-color: #000000; color: #ffffff; border-color: #000000;}"
    ".labeling-input .widget-label {font-weight: 700;}"
    "</style>"
)


class LabelingNotebookController:
    def __init__(
        self,
        *,
        engine,
        session: LabelingSession,
        universe: PreparedLabelingUniverse,
    ) -> None:
        self.engine = engine
        self.session = session
        self.config = session.config
        self.heatmap_mode = session.config.default_heatmap_mode
        self.frames = universe.frames
        self.windows_by_ticker = universe.windows_by_ticker
        self.current_frame = pd.DataFrame()
        self.context_frame = pd.DataFrame()
        self.current_window = None
        self.viewport_range = None
        self.loaded_positive_dates: set[pd.Timestamp] = set()
        self.working_positive_dates: set[pd.Timestamp] = set()
        self.figure = None

        self.status = widgets.HTML()
        self.hover_status = widgets.HTML()
        self.figure_output = widgets.Output()
        self.message_output = widgets.Output()
        self.timeframe = widgets.ToggleButtons(
            options=list(CHART_TIMEFRAME_SESSIONS),
            value=DEFAULT_CHART_TIMEFRAME,
            layout=widgets.Layout(width="auto", margin="0 0 0 24px"),
        )
        self.timeframe.style.button_width = "auto"
        self.timeframe.add_class("labeling-timeframe")
        self.heatmap = widgets.Dropdown(
            options=(
                ("Net return after commission", "net_return"),
                ("ATR units after commission", "atr_units"),
                ("Risk units after commission", "risk_units"),
            ),
            value=self.heatmap_mode,
            description="Heatmap",
        )
        self.heatmap.add_class("labeling-input")
        self.stop_multiple = widgets.BoundedFloatText(
            value=self.config.atr_stop_multiple,
            min=0.01,
            max=20.0,
            step=0.25,
            description="ATR stop",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="150px"),
        )
        self.stop_multiple.add_class("labeling-input")
        self.reward_risk = widgets.BoundedFloatText(
            value=self.config.reward_risk_ratio,
            min=0.01,
            max=20.0,
            step=0.25,
            description="Reward/risk",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="160px"),
        )
        self.reward_risk.add_class("labeling-input")
        self.reset_button = widgets.Button(
            description="Reset labels",
            button_style="warning",
            layout=widgets.Layout(width="auto"),
        )
        self.save_button = widgets.Button(
            description="Save",
            button_style="info",
            layout=widgets.Layout(width="auto"),
        )
        self.next_button = widgets.Button(
            description="Next",
            button_style="success",
            layout=widgets.Layout(width="auto"),
        )

        self.timeframe.observe(self._on_timeframe_change, names="value")
        self.heatmap.observe(self._on_visual_configuration_change, names="value")
        self.stop_multiple.observe(self._on_visual_configuration_change, names="value")
        self.reward_risk.observe(self._on_visual_configuration_change, names="value")
        self.reset_button.on_click(self._on_reset)
        self.save_button.on_click(self._on_save)
        self.next_button.on_click(self._on_next)

        self._load_current_window()

    def _refresh_chart_view(self) -> None:
        # Rebuild the context slice and initial viewport for the active window
        # and the currently selected timeframe.
        self.context_frame, self.viewport_range = prepare_chart_view(
            self.current_frame,
            window=self.current_window,
            config=self.config,
            timeframe=self.timeframe.value,
        )

    def _load_current_window(self) -> None:
        if self.session.completed:
            self.status.value = "<b>Labeling session complete.</b>"
            self.next_button.disabled = True
            self.save_button.disabled = True
            return
        ticker = self.session.current_ticker
        windows = self.windows_by_ticker[ticker]
        if not windows:
            raise ValueError(f"No complete labeling windows are available for {ticker}.")
        if self.session.current_window_position >= len(windows):
            raise ValueError(
                f"Stored window position is outside the available windows for {ticker}."
            )

        self.current_frame = self.frames[ticker]
        self.current_window = windows[self.session.current_window_position]
        # The figure centres on the planned window for the selected timeframe,
        # while this wider frame remains available for Plotly panning on either
        # side.
        self._refresh_chart_view()
        loaded = load_labels(
            engine=self.engine,
            provider=self.session.provider,
            ticker=ticker,
            start_date=self.current_window.start_date,
            end_date=self.current_window.end_date,
        )
        self.loaded_positive_dates = {trading_date for trading_date, label in loaded.items() if label}
        self.working_positive_dates = set(self.loaded_positive_dates)
        self._build_figure()
        self._update_status()

    def _build_figure(self) -> None:
        base = build_labeling_figure(
            self.context_frame,
            window=self.current_window,
            selected_dates=self.working_positive_dates,
            config=self.config,
            heatmap_mode=self.heatmap_mode,
            viewport_range=self.viewport_range,
        )
        self.figure = go.FigureWidget(base)
        self.figure._config = {**self.figure._config, "displaylogo": False}
        candle_trace = next(trace for trace in self.figure.data if trace.name == "OHLC")
        candle_trace.on_click(self._on_candle_click)
        candle_trace.on_hover(self._on_candle_hover)
        with self.figure_output:
            self.figure_output.clear_output(wait=True)
            display(self.figure)

    def _on_candle_click(self, trace, points, selector) -> None:
        del trace, selector
        if not points.point_inds:
            return
        point_position = points.point_inds[0]
        trading_date = pd.Timestamp(self.context_frame.index[point_position])
        if trading_date not in set(self.current_window.trading_dates):
            return
        if trading_date in self.working_positive_dates:
            self.working_positive_dates.remove(trading_date)
        else:
            self.working_positive_dates.add(trading_date)
        update_selected_trace(
            self.figure,
            self.context_frame,
            self.working_positive_dates,
        )
        self._update_status()

    def _on_candle_hover(self, trace, points, selector) -> None:
        del trace, selector
        if not points.point_inds:
            return
        trading_date = pd.Timestamp(self.context_frame.index[points.point_inds[0]])
        update_price_hud(self.figure, self.context_frame, trading_date=trading_date)
        guide = risk_guide_for_date(
            self.context_frame,
            trading_date=trading_date,
            config=self.config,
        )
        update_risk_guide_traces(self.figure, guide)
        if guide is None:
            self.hover_status.value = f"<b>{trading_date.date()}</b>: ATR unavailable"
            return
        self.hover_status.value = (
            f"<b>{trading_date.date()}</b> &nbsp; Close {guide.entry:.2f} &nbsp; "
            f"ATR {guide.atr:.2f} &nbsp; Stop {guide.stop:.2f} &nbsp; "
            f"Take profit {guide.take_profit:.2f}"
        )

    def _on_timeframe_change(self, change) -> None:
        del change
        self._refresh_chart_view()
        self._build_figure()

    def _on_visual_configuration_change(self, change) -> None:
        del change
        self.heatmap_mode = self.heatmap.value
        self.config = self.config.with_calibration(
            atr_stop_multiple=self.stop_multiple.value,
            reward_risk_ratio=self.reward_risk.value,
        )
        self._build_figure()

    def _on_reset(self, button) -> None:
        del button
        self.working_positive_dates = set(self.loaded_positive_dates)
        update_selected_trace(
            self.figure,
            self.context_frame,
            self.working_positive_dates,
        )
        self._update_status()

    def _save(self, *, advance: bool) -> None:
        next_ticker_position = None
        next_window_position = None
        completed = None
        if advance:
            ticker_windows = self.windows_by_ticker[self.session.current_ticker]
            if self.session.current_window_position + 1 < len(ticker_windows):
                next_ticker_position = self.session.current_ticker_position
                next_window_position = self.session.current_window_position + 1
                completed = False
            elif self.session.current_ticker_position + 1 < len(self.session.tickers):
                next_ticker_position = self.session.current_ticker_position + 1
                next_window_position = 0
                completed = False
            else:
                next_ticker_position = self.session.current_ticker_position
                next_window_position = self.session.current_window_position
                completed = True

        self.session = save_labeling_window(
            engine=self.engine,
            labeling_session_id=self.session.labeling_session_id,
            ticker=self.session.current_ticker,
            window=self.current_window,
            positive_dates=self.working_positive_dates,
            config=self.config,
            next_ticker_position=next_ticker_position,
            next_window_position=next_window_position,
            completed=completed,
        )
        self.loaded_positive_dates = set(self.working_positive_dates)
        if advance and not self.session.completed:
            self._load_current_window()
        elif self.session.completed:
            self.status.value = "<b>Labeling session complete.</b>"
            self.next_button.disabled = True
            self.save_button.disabled = True
        else:
            self._update_status(prefix="Saved. ")

    def _on_save(self, button) -> None:
        del button
        with self.message_output:
            self.message_output.clear_output(wait=True)
            try:
                self._save(advance=False)
            except Exception as error:
                print(f"Save failed: {error}")

    def _on_next(self, button) -> None:
        del button
        with self.message_output:
            self.message_output.clear_output(wait=True)
            try:
                self._save(advance=True)
            except Exception as error:
                print(f"Advance failed; the current window was not moved: {error}")

    def _update_status(self, *, prefix="") -> None:
        windows = self.windows_by_ticker[self.session.current_ticker]
        changed = self.working_positive_dates != self.loaded_positive_dates
        self.status.value = (
            f"{prefix}<b>{self.session.current_ticker}</b> &nbsp; "
            f"window {self.session.current_window_position + 1}/{len(windows)} &nbsp; "
            f"{self.current_window.start_date.date()}–{self.current_window.end_date.date()} &nbsp; "
            f"positive {len(self.working_positive_dates)}/{len(self.current_window.trading_dates)} &nbsp; "
            f"{'unsaved changes' if changed else 'saved state'}"
        )

    def display(self) -> None:
        controls_row = widgets.HBox(
            [self.reset_button, self.save_button, self.next_button, self.timeframe]
        )
        controls = widgets.VBox(
            [
                widgets.HBox([self.heatmap, self.stop_multiple, self.reward_risk]),
                controls_row,
                self.status,
                self.hover_status,
                self.message_output,
            ]
        )
        display(TIMEFRAME_BUTTON_STYLE, controls, self.figure_output)


controller = LabelingNotebookController(
    engine=engine,
    session=session,
    universe=universe,
)
controller.display()


## Persistence semantics

The label table has one authoritative row per `(provider, ticker, trading_date)`. `label_family` is metadata on that row, not part of the key. Saving an overlapping window updates the same rows rather than creating duplicates. Session progress and all labels for a window are committed in one transaction, so a failed **Next** cannot advance past unsaved work.